In [ ]:
from os import curdir
from math import e
import random
from IPython.display import clear_output
def clear():
    clear_output(wait=True)

# Classe para criar o jogador
class Player:
    def __init__(self,classChoice, life, proef, armorClass, att_dict, casterType):

      mod_dict = att_dict.copy()
      for att in att_dict:
        mod_dict[att] = (mod_dict[att] - 10) // 2

      self.name = classChoice
      self.mod_dict = mod_dict
      self.classChoice = classChoice
      self.maxLife = life + mod_dict['CON']
      self.currentLife = self.maxLife
      self.proef = proef
      self.armorClass = armorClass + mod_dict['DEX']
      self.att_dict = att_dict
      self.initiative = mod_dict['DEX']
      self.casterType = casterType
      if self.classChoice == 'Wizard':
        self.spellSaveDC = 10 + mod_dict['INT'] + self.proef
      self.exp = 0

    #Método para visualizar os status
    def showStats(self):
      #clear()
      print(f"Experience: {self.exp}")
      print(f"Class: {self.classChoice}")
      print(f"Life: {self.currentLife}/{self.maxLife}")
      print(f"Initiative: {self.mod_dict['DEX']}")
      print(f"Proeficence Bonus: {self.proef}")
      print(f"Armour Class: {self.armorClass}")
      print(
            f"STR: {self.att_dict['STR']} | "
            f"DEX: {self.att_dict['DEX']} | "
            f"CON: {self.att_dict['CON']} | "
            f"INT: {self.att_dict['INT']} | "
            f"WIS: {self.att_dict['WIS']} | "
            f"CHA: {self.att_dict['CHA']}"
           )
      print('Press any key to continue...')
      input(" ")

    # Método que recupera o personagem
    def longRest(self):
      print('You use long rest sucefully!')
      print(f"You have restore all your life points, abilitys and points!")
      self.currentLife = self.maxLife
      print('Press any key to continue...')
      input(" ")

    # Método que inicia
    def Start(self):
      while True:
        print(f"What you desire {self.classChoice}?")
        print('1) Long Rest')
        print('2) Show Stats')
        print('3) Combat')
        print('4) Exit')
        op = input("> ")
        match op:
          case '1':
            self.longRest()
          case '2':
            self.showStats()
          case '3':
            combat(self)
          case '4':
            print('Quiting...')
            break
          case _:
            print("Invalid choice!")

##########################################################
# ====----------        Let's Play!       ----------==== #
##########################################################

print('New game! Choose your attributes:')
print('You have this points to apply for your character:')

# ================ Choose your points ================== #

# Atributos para serem comprados
attKey = ['STR', 'DEX', 'CON', 'INT', 'WIS', 'CHA']
valorPoints = [16, 14, 14, 12, 10, 8]
#Cria um dicionário vazio, que vai usar as chaves de "attKey" e o valor de "valorPoints"
att_dict = {}

check = False
while check == False:
  for att in attKey:
        while True: # Checa se os valores são preenchidos corretamente
          # imprime os valores disponiveis
          print('Available:', valorPoints)
          # att percorre a lista attKey, imprimindo a chhave atual
          valor = int(input(f'{att}: '))

          #checa se o valor está no valorPoints
          if valor in valorPoints:
              # Diz que na chave atual de "att" coloca o valor escolhido
              att_dict[att] = valor
              # remove o valor escolhido de "valorPoints"
              valorPoints.remove(valor)
              break
          else:
              print('Invalid choice!')

  print('Are you sure of your points?')
  print(att_dict)
  while True:
    print('Y/N?')
    op = input("> ").upper()
    match op:
      case 'Y':
        check = True
        break
      case 'N':
        attKey = ['STR', 'DEX', 'CON', 'INT', 'WIS', 'CHA']
        valorPoints = [16, 14, 14, 12, 10, 8]
        att_dict = {}
        clear()
        break
      case _:
        print("Invalid choice!")

# ================= Choose your class ================== #

while True:
  print('1) Wizard 1d6 life')
  print('2) Fighter 1d10 life')
  op = input("> ")
  match op:
    case '1':
      classChoice = 'Wizard'
      newChar = Player(classChoice, 6, 2, 10, att_dict, 'caster')
      break
    case '2':
      classChoice = 'Fighter'
      newChar = Player(classChoice, 10, 2, 10, att_dict, 'martial')
      break
    case _:
      print("Invalid choice!")

##########################################################
# ====----------          Combat!         ----------==== #
##########################################################

# Função para ver se o dano foi crítico
def criticRoll(roll, damage):
    if roll == 20:
        print('YOU GOT A CRIT!')
        return damage * 2
    return damage

# Função para ver se acertou
def checkArmor(roll, target):
    if roll >= target.armorClass:
        print(f"{target.name} was hit!")
        return True
    else:
        print(f"{target.name} dodged!")
        return False

# Função para se passou no teste
def saveDex(target, dc):
    save = random.randint(1, 20) + target.mod_dict['DEX']
    return save >= dc

# ====----------      Fighter Actions     ----------==== #

def fighterActions(player, enemy):
  cooldown = 0
  while True:
    print(f"What you desire {player.classChoice}?")
    print(f"You have {player.currentLife}/{player.maxLife}")
    print(f"1) Long Sword")
    print(f"2) Long Bow ")
    print(f"3) Second Wind: {cooldown}")
    print(f"4) Flee")
    op = input("> ")

    match op:
      case '1':
        print('You use Long Sword!')
        d20 = random.randint(1, 20)
        attack = d20 + player.mod_dict['STR'] + player.proef
        hit = checkArmor(attack, enemy)
        if hit:
            damage = random.randint(1, 12) + player.mod_dict['STR']
            damage = criticRoll(d20, damage)
            enemy.currentLife -= damage
            print(f"You rolled {attack} and dealt {damage} slashing damage!")
        else:
            print("You missed!")
        return
      case '2':
        #clear()
        print('You use Long Bow!')
        d20 = random.randint(1, 20)
        attack = d20 + player.mod_dict['DEX'] + player.proef
        hit = checkArmor(attack, enemy)
        if hit:
          damage = random.randint(1, 8) + player.mod_dict['DEX']
          damage = criticRoll(d20, damage)
          enemy.currentLife -= damage
          print(f"You rolled {attack} and dealt {damage} piercing damage!")
        else:
          print("You missed!")
        return
      case '3':
        if cooldown == 0:
          #clear()
          print('You use Second Wind!')
          roll = random.randint(1, 10) + player.mod_dict['CON']
          player.currentLife = min(player.currentLife + roll, player.maxLife)
          print(f"You heal {roll}!")
          cooldown = 8
        else:
          print("Ability on cooldown!")
        return
      case '4':
        #clear()
        print('When you try to flee, your enemy gain a attack. Are your sure?')
        print('Y/N?')
        op = input("> ").upper()
        if op == 'Y':
          enemy.attack(player)
          print("You fled from combat, for now...")
          return "flee"
          break
        if op == 'N':
          print("You didn't flee!")
          return
      case _:
        print(f"Invalid choice {player.classChoice}!")
        break

# ====----------       Wizard Actions     ----------==== #

def wizardActions(player, enemy):
  cooldown = 0
  while True:

    print(f"What you desire {player.classChoice}?")
    print(f"You have {player.currentLife}/{player.maxLife}")
    print(f"1) Fire bolt")
    print(f"2) Staff atack")
    print(f"3) Burning Hands Cooldown: {cooldown}")
    print(f"4) Flee")
    op = input("> ")

    match op:
      case '1':
        #clear()
        print('You use Fire Bolt!')
        d20 = random.randint(1, 20)
        attack = d20 + player.mod_dict['INT'] + player.proef
        hit = checkArmor(attack, enemy)
        if hit:
          damage = random.randint(1, 10) + player.mod_dict['INT']
          damage = criticRoll(d20, damage)
          enemy.currentLife -= damage
          print(f"You rolled {attack} and dealt {damage} fire damage!")
        else:
          print("You missed!")
        return
      case '2':
        #clear()
        print('You use Staff Atack')
        d20 = random.randint(1, 20)
        attack = d20 + player.mod_dict['STR']
        hit = checkArmor(attack, enemy)
        if hit:
          damage = random.randint(1, 8) + player.mod_dict['STR']
          damage = criticRoll(d20, damage)
          enemy.currentLife -= damage
          print(f"You rolled {attack} and dealt {damage} bludgeoning damage!")
        else:
          print("You missed!")
        return
      case '3':
        #clear()
        if cooldown == 0:
          print('You use Burning Hands')
          damage_rolls = [random.randint(1, 6) for _ in range(3)]
          total_damage = sum(damage_rolls)
          save = saveDex(enemy, player.spellSaveDC)
          print(f"Damage rolls: {damage_rolls}")
          if save:
            print(f"{enemy.name} passed the save!")
            total_damage = total_damage // 2
            enemy.currentLife -= total_damage
            print(f"Total damage: {total_damage}")
            cooldown = 3
          else:
            print(f"{enemy.name} failed the save!")
            enemy.currentLife -= total_damage
            print(f"Total damage: {total_damage}")
            cooldown = 3
        else:
          print("Spell on cooldown!")
        return
      case '4':
        #clear()
        print('When you try to flee, your enemy gain a attack. Are your sure?')
        print('Y/N?')
        op = input("> ").upper()
        if op == 'Y':
          enemy.attack(player)
          print("You fled from combat, for now...")
          return "flee"
          break
        if op == 'N':
          print("You didn't flee!")
          return
      case _:
        print(f"Invalid choice {player.classChoice}!")
        break
      # reduz cooldown
    if cooldown > 0:
      cooldown -= 1

# ====----------       Enemys  Stats!     ----------==== #

class Enemy:
    def __init__(self, name, life, armorClass, mod_dict, proef):
        self.name = name
        self.life = life
        self.currentLife = life
        self.armorClass = armorClass
        self.mod_dict = mod_dict
        self.proef = proef
        self.initiative = mod_dict['DEX']

class Goblin(Enemy):
    def __init__(self):
        mod_dict = {
            'STR': -1,
            'DEX': 2,
            'CON': 0,
            'INT': 0,
            'WIS': -1,
            'CHA': -1
        }
        life = sum([random.randint(1, 6) for _ in range(2)])

        super().__init__(
            name="Goblin",
            life=life,
            armorClass=10 + mod_dict['DEX'],
            mod_dict=mod_dict,
            proef=2
        )

    def attack(self, player):
        print(f"{self.name} tries to stab you!")

        d20 = random.randint(1, 20)
        attack = d20 + self.mod_dict['DEX'] + self.proef

        hit = checkArmor(attack, player)

        if hit:
            damage = random.randint(1, 4) + self.mod_dict['DEX']
            damage = criticRoll(d20, damage)
            player.currentLife -= damage
            print(f"{self.name} dealt {damage} damage!")
        else:
            print(f"{self.name} missed!")

# ====----------       Combat  Turns!     ----------==== #

def combat(player):
    # Checa os inimigos que aparecem a depender do nível
    if player.exp >= 0 and player.exp <= 3:
        enemy = Goblin()

    print(f"Let's go {player.classChoice}, you will fight a {enemy.name}!")

    # ? iniciativa do player >= inimigo
    player_turn = player.initiative >= enemy.initiative

    while player.currentLife > 0 and enemy.currentLife > 0:

        if player_turn:
            print("\n--- YOUR TURN ---")

            if player.classChoice == 'Wizard':
                result = wizardActions(player, enemy)
            elif player.classChoice == 'Fighter':
                result = fighterActions(player, enemy)

            if result == "flee":
                print("Back to menu...")
                return

        else:
            print("\n--- ENEMY TURN ---")
            # Chama o método do Enemy attack, e passa o player como parâmetro
            enemy.attack(player)

        # alterna turno
        player_turn = not player_turn

    # fim do combate
    if player.currentLife <= 0:
        print("You died...")
    else:
        print(f"You defeated the {enemy.name}!")

# ====----------        Let's Start!      ----------==== #

newChar.Start()

New game! Choose your attributes:
You have this points to apply for your character:
Available: [16, 14, 14, 12, 10, 8]
Available: [16, 14, 14, 12, 10]
Available: [16, 14, 12, 10]
Available: [16, 12, 10]
Available: [12, 10]
Available: [12]
Are you sure of your points?
{'STR': 8, 'DEX': 14, 'CON': 14, 'INT': 16, 'WIS': 10, 'CHA': 12}
Y/N?
1) Wizard 1d6 life
2) Fighter 1d10 life
What you desire Wizard?
1) Long Rest
2) Show Stats
3) Combat
4) Exit
You use long rest sucefully!
You have restore all your life points, abilitys and points!
Press any key to continue...
What you desire Wizard?
1) Long Rest
2) Show Stats
3) Combat
4) Exit
Experience: 0
Class: Wizard
Life: 8/8
Initiative: 2
Proeficence Bonus: 2
Armour Class: 12
STR: 8 | DEX: 14 | CON: 14 | INT: 16 | WIS: 10 | CHA: 12
Press any key to continue...
What you desire Wizard?
1) Long Rest
2) Show Stats
3) Combat
4) Exit
Let's go Wizard, you will fight a Goblin!

--- YOUR TURN ---
What you desire Wizard?
You have 8/8
1) Fire bolt
2) Staff 